In [1]:
import pandas as pd


In [2]:
import numpy as np
import pickle

In [3]:
import tensorflow as tf
from tensorflow.keras.models import load_model

In [5]:
## Load the trained model, scaler, geo and gender pickle files
model = load_model("model.h5")

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)




In [6]:
## Example input data

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure':3,
    'Balance': 60000,
    'NumOfProducts':2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000

}

In [7]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()

c:\Users\raviv\Documents\ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [8]:
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [9]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [ ]:
input_data = pd.concat([input_data.reset_index(drop=True),geo_encoded_df],axis=1)

AttributeError: 'dict' object has no attribute 'reset_index'

In [13]:
input_df = pd.DataFrame([input_data])

In [15]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [18]:
input_df['Geography'] = onehot_encoder_geo.transform([input_df['Geography']])

c:\Users\raviv\Documents\ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


TypeError: sparse array length is ambiguous; use getnnz() or shape[0]

In [19]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [20]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [21]:
## Concatenate the Geography column
input_df = pd.concat([input_df.drop("Geography", axis=1),geo_encoded_df],axis=1)

In [22]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [23]:
## Scaling the data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [24]:
## Predict Churn
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step


array([[0.03241925]], dtype=float32)

In [26]:
prediction_proba = prediction[0][0]

In [27]:
prediction_proba

np.float32(0.032419253)

In [28]:
if prediction_proba > 0.5:
    print("The customer is likely to churn")
else:
    print("The customer is not likely to churn")

The customer is not likely to churn
